In [24]:
DATA_DIR = "../data"
MODELS_DIR = "../models"

In [25]:
import pandas as pd

df = pd.read_csv(f"{DATA_DIR}/raw/dataset.csv")
df["date"] = pd.to_datetime(df["date"])

display(df)

,Unnamed: 0.1,Unnamed: 0,date,team_1,team_2,_map,result_1,result_2,map_winner,starting_ct,...,t_2,t_1,ct_2,event_id,match_id,rank_1,rank_2,map_wins_1,map_wins_2,match_winner
0,0,0,2020-03-18,Recon 5,TeamOne,Dust2,0,16,2,2,...,1,0,15,5151,2340454,62,63,0,2,2
1,1,1,2020-03-18,Recon 5,TeamOne,Inferno,13,16,2,2,...,6,5,10,5151,2340454,62,63,0,2,2
2,2,2,2020-03-18,New England Whalers,Station7,Inferno,12,16,2,1,...,6,3,10,5243,2340461,140,118,12,16,2
3,3,3,2020-03-18,Rugratz,Bad News Bears,Inferno,7,16,2,2,...,8,7,8,5151,2340453,61,38,0,2,2
4,4,4,2020-03-18,Rugratz,Bad News Bears,Vertigo,8,16,2,2,...,5,4,11,5151,2340453,61,38,0,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45767,45767,45768,2015-11-05,G2,E-frag.net,Inferno,13,16,2,1,...,7,5,9,1970,2299059,7,16,1,2,2
45768,45768,45769,2015-11-05,G2,E-frag.net,Dust2,16,13,1,1,...,5,6,8,1970,2299059,7,16,1,2,2
45769,45769,45770,2015-11-04,CLG,Liquid,Inferno,16,12,1,1,...,8,9,4,1934,2299011,10,14,16,12,1
45770,45770,45771,2015-11-03,NiP,Dignitas,Train,16,4,1,2,...,1,12,3,1934,2299001,6,12,16,4,1


In [26]:
df_test = pd.read_csv(f"{DATA_DIR}/preprocessed/test.csv")

display(df_test)

,elo_diff,winrate_10_diff,winrate_30_diff,experience_diff,rank_diff,h2h_winrate,team_1_wins
0,1.139435,0.231677,0.815696,-0.003897,-0.967555,-0.076338,1
1,-0.850737,-1.735120,-2.123590,-0.155839,3.686040,-0.076338,0
2,0.040321,0.161434,-0.154268,-0.231810,-0.578346,-0.076338,1
3,0.945895,1.074590,-1.006661,2.989363,-0.256825,-0.076338,1
4,-1.066911,-0.892207,0.420992,-2.475489,3.110686,-0.076338,0
...,...,...,...,...,...,...,...
4566,-0.608141,0.793619,-0.653947,-0.201422,0.538517,-1.544189,0
4567,-0.758610,0.793619,-0.653947,-0.201422,0.538517,-1.544189,0
4568,-0.364520,0.512648,-0.623276,-0.003897,0.521595,1.391514,0
4569,0.093123,0.512648,0.404196,-2.597043,0.132385,-0.076338,0


In [27]:
from keras.models import load_model

model = load_model(f"{MODELS_DIR}/model.keras")

display(model.summary())  # type: ignore

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,157 (20.15 KB)

 Trainable params: 1,665 (6.50 KB)

 Non-trainable params: 160 (640.00 B)

 Optimizer params: 3,332 (13.02 KB)

None

In [28]:
import joblib

preprocessor = joblib.load(f"{MODELS_DIR}/preprocessor.joblib")

In [29]:
# Performance on the test dataset
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score

target = "team_1_wins"

X_test = df_test.drop(target, axis=1)
y_test = df_test[target]

y_proba = model.predict(X_test).ravel()  # type: ignore
y_pred = (y_proba >= 0.5).astype(int)

df_test_predicted = pd.concat([df_test, pd.Series(y_pred, name="prediction")], axis=1)

display(df_test_predicted)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_proba))
print("Brier score:", brier_score_loss(y_test, y_proba))

143/143 ━━━━━━━━━━━━━━━━━━━━ 0s 334us/step


,elo_diff,winrate_10_diff,winrate_30_diff,experience_diff,rank_diff,h2h_winrate,team_1_wins,prediction
0,1.139435,0.231677,0.815696,-0.003897,-0.967555,-0.076338,1,1
1,-0.850737,-1.735120,-2.123590,-0.155839,3.686040,-0.076338,0,0
2,0.040321,0.161434,-0.154268,-0.231810,-0.578346,-0.076338,1,1
3,0.945895,1.074590,-1.006661,2.989363,-0.256825,-0.076338,1,1
4,-1.066911,-0.892207,0.420992,-2.475489,3.110686,-0.076338,0,0
...,...,...,...,...,...,...,...,...
4566,-0.608141,0.793619,-0.653947,-0.201422,0.538517,-1.544189,0,0
4567,-0.758610,0.793619,-0.653947,-0.201422,0.538517,-1.544189,0,0
4568,-0.364520,0.512648,-0.623276,-0.003897,0.521595,1.391514,0,1
4569,0.093123,0.512648,0.404196,-2.597043,0.132385,-0.076338,0,1


Accuracy: 0.7611026033690659
AUC: 0.8430766383534918
Brier score: 0.16065412759780884


In [30]:
def build_feature_state(df, base_elo=1500):
    """Build initial state from historical matches"""
    import pandas as pd  # type: ignore

    state = {
        "elo": {},
        "matches_played": {},
        "win_history": {},
        "h2h": {},
    }
    # Initialize Elo
    teams = pd.concat([df["team_1"], df["team_2"]]).unique()
    for team in teams:
        state["elo"][team] = base_elo
        state["matches_played"][team] = 0
        state["win_history"][team] = []
    return state

def compute_features_for_match(match, state):
    """Compute feature vector for a single match"""
    import numpy as np  # type: ignore

    team = match["team_1"]
    opp = match["team_2"]

    # Elo diff
    elo_team = state["elo"].get(team, 1500)
    elo_opp = state["elo"].get(opp, 1500)
    elo_diff = elo_team - elo_opp

    # Rolling winrates
    winrate_10 = (
        np.mean(state["win_history"].get(team, [])[-10:])
        if len(state["win_history"].get(team, [])) > 0
        else 0
    )
    winrate_30 = (
        np.mean(state["win_history"].get(team, [])[-30:])
        if len(state["win_history"].get(team, [])) > 0
        else 0
    )
    winrate_10_diff = winrate_10 - (
        np.mean(state["win_history"].get(opp, [])[-10:])
        if len(state["win_history"].get(opp, [])) > 0
        else 0
    )
    winrate_30_diff = winrate_30 - (
        np.mean(state["win_history"].get(opp, [])[-30:])
        if len(state["win_history"].get(opp, [])) > 0
        else 0
    )

    # Experience / matches played
    experience_diff = state["matches_played"].get(team, 0) - state[
        "matches_played"
    ].get(opp, 0)

    # Rank diff
    rank_diff = match["rank_1"] - match["rank_2"]

    # H2H winrate
    h2h_key = (team, opp)
    h2h_list = state["h2h"].get(h2h_key, [])
    h2h_winrate = np.mean(h2h_list) if len(h2h_list) > 0 else 0.5

    return {
        "elo_diff": elo_diff,
        "winrate_10_diff": winrate_10_diff,
        "winrate_30_diff": winrate_30_diff,
        "experience_diff": experience_diff,
        "rank_diff": rank_diff,
        "h2h_winrate": h2h_winrate,
    }

def update_state_with_result(match, state, k=32):
    """Update Elo, rolling winrates, H2H after a match"""
    team = match["team_1"]
    opp = match["team_2"]
    result = int(match["match_winner"] == 1)

    # Update Elo
    r_team = state["elo"].get(team, 1500)
    r_opp = state["elo"].get(opp, 1500)
    exp = 1 / (1 + 10 ** ((r_opp - r_team) / 400))
    state["elo"][team] = r_team + k * (result - exp)
    state["elo"][opp] = r_opp + k * ((1 - result) - (1 - exp))

    # Update win history
    state["win_history"].setdefault(team, []).append(result)
    state["win_history"].setdefault(opp, []).append(1 - result)

    # Update matches played
    state["matches_played"][team] = state["matches_played"].get(team, 0) + 1
    state["matches_played"][opp] = state["matches_played"].get(opp, 0) + 1

    # Update H2H
    h2h_key = (team, opp)
    state["h2h"].setdefault(h2h_key, []).append(result)
    return state



def match_to_features(preprocessor, history_df, match):
    """
    match: dict with team_1, team_2, rank_1, rank_2, date
    """
    import pandas as pd  # type: ignore

    # Build state from past only
    past = history_df[history_df["date"] < match["date"]]
    state = build_feature_state(past)

    # Replay past matches to update state
    for _, m in past.sort_values("date").iterrows():
        update_state_with_result(m, state)

    # Compute features for the future match
    X = pd.DataFrame([compute_features_for_match(match, state)])

    X_scaled = preprocessor.transform(X)

    return X_scaled

# Do a test prediction on a made up match
future_match = {
    "team_1": "Rugratz",
    "team_2": "Bad News Bears",
    "rank_1": 61,
    "rank_2": 38,
    "date": pd.Timestamp("2024-06-01"),
}

X = match_to_features(preprocessor, df, future_match)
p = model.predict(X)[0][0]  # type: ignore

print(f"{future_match['team_1']} win probability: {p:.2%}")
print(
    "Predicted winner:", future_match["team_1"] if p > 0.5 else future_match["team_2"]
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
Rugratz win probability: 18.13%
Predicted winner: Bad News Bears
